# Problem 2: SynCode with Phi-2

In [1]:
%pip install -q --ignore-requires-python git+https://github.com/uiuc-focal-lab/syncode.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Grammar: the model must output "Operand1: <float> Operand2: <float> Operator: <op>"
grammar = r"""
start: "Operand1:" NUMBER "Operand2:" NUMBER "Operator:" OP

NUMBER: /[0-9]+\.[0-9]+/
OP: "+" | "-" | "*" | "/"

%ignore /[ \t\n]+/
"""

# Few-shot examples (extraction only, no final answers)
few_shot = """Extract the two operands (as floats) and the operator from each query.

Query: What is 12.5 plus 7.25?
Operand1: 12.5 Operand2: 7.25 Operator: +

Query: What is 9. minus 2.5?
Operand1: 9.0 Operand2: 2.5 Operator: -

Query: Multiply 6.0 by 8 for me.
Operand1: 6.0 Operand2: 8.0 Operator: *

Query: What do you get when 81.6 is divided by 3.0?
Operand1: 81.6 Operand2: 3.0 Operator: /

"""

In [3]:
from syncode import Syncode

syn_llm = Syncode(
    model="microsoft/phi-2",
    mode="grammar_strict",
    grammar=grammar,
    parse_output_only=True,
    max_new_tokens=60,
)

[2026-10-29 05:10:59,712-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-29 05:12:56,052-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_1030918380_50257.pkl exist: False
[2026-10-29 05:12:56,053-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_1030918380_50257.pkl.
[2026-10-29 05:12:56,552-syncode.mask_store.fsm_set] - 6 FSMs with 44 states initialized in 0.00 seconds
[2026-10-29 05:12:56,554-syncode.mask_store.mask_store] - Ignore whitespace is True
[2026-10-29 05:12:56,564-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 36 to 11


100%|██████████| 44/44 [00:07<00:00,  5.56it/s]

[2026-10-29 05:13:04,529-syncode.mask_store.mask_store] - Time taken to create mask store: 8.47 seconds


In [4]:
import re

def run_syncode(textual_query):
    # extract the operands and operator
    prompt = few_shot + "Query: " + textual_query + "\n"
    output = syn_llm.infer(prompt)[0]

    match = re.search(r"Operand1:\s*(\S+)\s*Operand2:\s*(\S+)\s*Operator:\s*(\S)", output)
    a = float(match.group(1))
    b = float(match.group(2))
    op = match.group(3)

    if op == "+":
        answer = a + b
    elif op == "-":
        answer = a - b
    elif op == "*":
        answer = a * b
    else:
        answer = a / b

    return [a, b], op, round(answer, 10)

In [5]:
queries = [
    "What is 327. multiplied by 11.0?",
    "What is 45.1 plus 23.54?",
    "What is 120.4 divided by 4.0?",
]

for q in queries:
    print(q)
    print(run_syncode(q))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is 327. multiplied by 11.0?
([327.0, 11.0], '*', 3597.0)
What is 45.1 plus 23.54?
([45.1, 23.54], '+', 68.64)
What is 120.4 divided by 4.0?
([120.4, 4.0], '/', 30.1)
